In [8]:
import boto3
import sagemaker
from sagemaker.estimator import Estimator


# ============================================================
# 기본 설정
# ============================================================
session    = boto3.Session()
sm_session = sagemaker.Session(boto_session=session)

REGION     = session.region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# ============================================================
# 경로 설정
# ============================================================
IMAGE_URI         = "155954279556.dkr.ecr.us-east-1.amazonaws.com/gs-automl-base-containers/boilerplate312:sample-v1.0"
SAGEMAKER_ROLE    = sagemaker.get_execution_role()

CONF_S3_PATH      = "s3://gs-retail-awesome-conf-us-east-1/dev/sample/titanic-survival-prediction/baseline-v1/"
NOTEBOOK_S3_PATH  = "s3://gs-retail-awesome-conf-us-east-1/dev/sample/titanic-survival-prediction/baseline-v1/titanic_modeling.ipynb"
WORK_DIR          = "/opt/ml/code"



INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


In [9]:
# ============================================================
# 노트북 S3 업로드 (Estimator 실행 전)
# ============================================================
from pathlib import Path
from urllib.parse import urlparse

LOCAL_NOTEBOOK_PATH = Path("/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sample/run_pm_2/modeling/titanic_modeling.ipynb")

def upload_notebook_to_s3(local_path: Path, s3_uri: str) -> None:
    """로컬 노트북을 S3에 업로드"""
    parsed = urlparse(s3_uri)
    bucket = parsed.netloc
    key    = parsed.path.lstrip("/")

    if not local_path.exists():
        raise FileNotFoundError(f"Notebook not found: {local_path}")

    s3_client = boto3.client("s3", region_name=REGION)
    s3_client.upload_file(str(local_path), bucket, key)
    print(f"✅ Notebook uploaded")
    print(f"   Local : {local_path}")
    print(f"   S3    : {s3_uri}")

upload_notebook_to_s3(LOCAL_NOTEBOOK_PATH, NOTEBOOK_S3_PATH)

✅ Notebook uploaded
   Local : /home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sample/run_pm_2/modeling/titanic_modeling.ipynb
   S3    : s3://gs-retail-awesome-conf-us-east-1/dev/sample/titanic-survival-prediction/baseline-v1/titanic_modeling.ipynb


In [11]:
# ============================================================
# Estimator 설정
# ============================================================
estimator = Estimator(
    base_job_name  = "boilerplate312-sm-sample",
    image_uri      = IMAGE_URI,
    entry_point    = "run_pm.py",
    role           = SAGEMAKER_ROLE,
    instance_type  = "ml.m5.xlarge",
    instance_count = 1,
    sagemaker_session = sm_session,
    hyperparameters = {
        "conf-s3-path"  : CONF_S3_PATH,
        "work-dir"      : WORK_DIR,
    },
)

# ============================================================
# 학습 실행
# ============================================================
estimator.fit(wait=True, logs="All")

INFO:sagemaker:Creating training-job with name: boilerplate312-sm-sample-2026-03-17-02-26-08-236


2026-03-17 02:26:09 Starting - Starting the training job...
2026-03-17 02:26:23 Starting - Preparing the instances for training...
2026-03-17 02:27:02 Downloading - Downloading the training image...
2026-03-17 02:27:28 Training - Training image download completed. Training in progress..2026-03-17 02:27:37,616 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-17 02:27:37,617 sagemaker-training-toolkit INFO     Failed to parse hyperparameter conf-s3-path value s3://gs-retail-awesome-conf-us-east-1/dev/sample/titanic-survival-prediction/baseline-v1/ to Json.
Returning the value itself
2026-03-17 02:27:37,617 sagemaker-training-toolkit INFO     Failed to parse hyperparameter work-dir value /opt/ml/code to Json.
Returning the value itself
2026-03-17 02:27:37,617 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-03-17 02:27:37,631 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus instal